# Dataset B GSE249479 Phase 1: metadata, replication, and memory audit

This notebook performs only the Phase 1 audit of the configured public GSE249479 H5AD object. It does not run PCA, Scanorama, Harmony, scVI, UMAP, ScGeo, RNA velocity, or comparator analysis.

Scientific and resource boundaries enforced here: no ScGeo package changes, no frozen threshold tuning, no cells-as-replicates inference, no full expression matrix densification, sparse matrix storage is preserved, RSS is recorded around each major section, and execution stops cleanly if RSS exceeds the configured threshold.

## Reader guide

- **Purpose:** Dataset B GSE249479 Phase 1: metadata, replication, and memory audit in the frozen revision workflow.
- **Inference scope:** Dataset B is descriptive_only because no valid biological-replicate identity was recovered. Cells, conditions, libraries, and inferred partitions are not replicates.
- **Inputs:** Official study metadata and the immutable workstation H5AD, where applicable.
- **Implementation:** `scripts/gse249479_memory_safe.py`.
- **Outputs:** Ignored `results/public_validation/gse249479_dataset_b/` artifacts.
- **Frozen findings:** Retain the accepted descriptive TNF/LPS geometry, representation sensitivity, signature, abundance/distribution, and official R Augur evidence. The Python comparator remains supplementary only.
- **Limitations:** No population-level biological inference is available; nested PCA views are not independent confirmations and UMAP is display-only.

This source notebook is intentionally a thin, output-free entry point. The testable implementation is maintained in scripts/gse249479_memory_safe.py. Executed review copies and generated artifacts are written under the ignored results directory.


In [ ]:
from pathlib import Path
import gc
import os
import sys

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "scripts"))
from gse249479_memory_safe import (
    MemoryAudit,
    MemoryLimitExceeded,
    bytes_to_gb,
    candidate_replicate_audit,
    configured_paths,
    ensure_output_tree,
    feasibility_assessment,
    inspect_h5ad_storage,
    inspect_obs_metadata,
    load_config,
    memory_threshold_bytes,
    relative_or_absolute,
    replication_decision,
    require_active_branch,
    rss_bytes,
    source_file_provenance,
    write_json,
    write_metadata,
)

import pandas as pd

CONFIG = load_config(ROOT)
PATHS = configured_paths(CONFIG, ROOT)
OUTPUT_DIR = PATHS["output_dir"]
INPUT_H5AD = PATHS["input_h5ad"]
ensure_output_tree(OUTPUT_DIR)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / "_matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
AUDIT = MemoryAudit(memory_threshold_bytes(CONFIG))

In [ ]:
def flatten_matrix_record(label, info):
    out = {"matrix": label}
    for key, value in info.items():
        if isinstance(value, (tuple, list)):
            out[key] = "x".join(str(x) for x in value)
        elif isinstance(value, dict):
            continue
        else:
            out[key] = value
    for key in [
        "data_bytes",
        "indices_bytes",
        "indptr_bytes",
        "estimated_sparse_bytes",
        "estimated_dense_bytes",
        "estimated_dense_float32_bytes",
        "estimated_dense_float64_bytes",
    ]:
        if key in out and pd.notna(out[key]):
            out[f"{key}_gb"] = bytes_to_gb(out[key])
    return out

def write_csv(name, frame):
    path = OUTPUT_DIR / "audit" / f"{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(f".{path.name}.tmp.{os.getpid()}")
    frame.to_csv(tmp_path, index=False)
    os.replace(tmp_path, path)
    return path

def empty_candidate_fields():
    return pd.DataFrame(columns=[
        "field_name",
        "field_category",
        "n_unique_values",
        "values_per_condition",
        "missing_fraction",
        "nested_or_confounded_with_condition",
        "final_eligibility_decision",
        "eligibility_reason",
    ])

def placeholder_matrix(status, reason):
    return {
        "key": "X",
        "present": False,
        "storage": None,
        "sparse_format": None,
        "encoding_type": None,
        "shape": None,
        "dtype": None,
        "nnz": None,
        "estimated_sparse_bytes": None,
        "estimated_dense_bytes": None,
        "estimated_dense_float32_bytes": None,
        "estimated_dense_float64_bytes": None,
        "property_status": status,
        "reason": reason,
    }

storage = None
obs_metadata = None
candidate_fields = empty_candidate_fields()
replication = None
feasibility = None
metadata_audit = {}
stopped_cleanly = False
stop_reason = ""
branch = None
provenance = {}
storage_error = None
obs_error = None

try:
    with AUDIT.section("branch_guard"):
        branch = require_active_branch(ROOT, CONFIG["required_git_branch"])

    with AUDIT.section("source_file_provenance"):
        provenance = source_file_provenance(INPUT_H5AD, ROOT)
        provenance.update({
            "active_branch": branch,
            "required_branch": CONFIG["required_git_branch"],
            "phase": CONFIG["phase"],
            "immutable_source_policy": "The source H5AD is opened read-only with h5py or anndata backed='r'; this workflow never writes back to the source path.",
            "forbidden_phase1_steps": CONFIG["forbidden_phase1_steps"],
        })
        write_json(OUTPUT_DIR / "audit" / "provenance.json", provenance)
        write_csv("00_dataset_file_status", pd.DataFrame([{
            "input_h5ad": provenance["absolute_path"],
            "input_h5ad_display": provenance["display_path"],
            "exists": provenance["exists"],
            "file_size_bytes": provenance["file_size_bytes"],
            "file_size_gb": provenance["file_size_gb"],
            "modified_time_utc": provenance["modified_time_utc"],
            "sha256": provenance["sha256"],
            "immutable_source": provenance["immutable_source"],
            "status": "available" if provenance["exists"] else "missing_input",
        }]))

    if not provenance.get("exists", False):
        stopped_cleanly = True
        stop_reason = "Configured input H5AD was not found; downstream Phase 1 sections were not attempted."
        x_info = placeholder_matrix("missing_input", stop_reason)
        write_csv("00_matrix_storage_audit", pd.DataFrame([flatten_matrix_record("X", x_info)]))
        write_csv("00_raw_and_layers_availability", pd.DataFrame([{
            "raw_present": False,
            "raw_keys": "",
            "n_layers": 0,
            "layer_names": "",
            "status": "missing_input",
        }]))
        write_csv("obs_columns", pd.DataFrame(columns=["column", "position"]))
        write_csv("00_obs_columns", pd.DataFrame(columns=["column", "position"]))
        write_csv("00_var_columns", pd.DataFrame(columns=["column", "position"]))
        write_csv("00_metadata_columns_summary", pd.DataFrame(columns=["column", "dtype", "n_missing", "n_unique", "example_values"]))
        write_csv("condition_counts", pd.DataFrame(columns=["column", "value", "n_cells"]))
        write_csv("00_condition_counts", pd.DataFrame(columns=["column", "value", "n_cells"]))
        write_csv("00_qc_fields", pd.DataFrame(columns=["column"]))
        write_csv("candidate_replicate_fields", candidate_fields)
        write_csv("00_replicate_and_clade_metadata", pd.DataFrame([
            {"metadata_type": "biological_donor", "columns": ""},
            {"metadata_type": "xenograft_recipient", "columns": ""},
            {"metadata_type": "source_pool", "columns": ""},
            {"metadata_type": "genetic_clade", "columns": ""},
            {"metadata_type": "experimental_library", "columns": ""},
            {"metadata_type": "condition", "columns": ""},
            {"metadata_type": "inferred_cell_label", "columns": ""},
        ]))
        replication = {
            "valid_biological_replicate_units": False,
            "decision": "input_missing",
            "reason": "The configured H5AD file is absent, so replicate metadata cannot be evaluated. Cells must not be treated as biological replicates.",
        }
        write_csv("00_replication_decision", pd.DataFrame([replication]))
        feasibility = pd.DataFrame([
            {"step": step, "phase1_status": "not_run", "estimated_feasibility": "input_missing", "memory_basis": "No matrix audit available", "required_next_action": "Provide the configured public H5AD object and rerun notebook 00."}
            for step in ["PCA", "Scanorama", "Harmony", "scVI"]
        ])
        write_csv("memory_feasibility", feasibility)
        write_csv("00_feasibility_assessment", feasibility)
        metadata_audit = {
            "status": "missing_input",
            "input_h5ad": provenance,
            "shape": None,
            "matrix": flatten_matrix_record("X", x_info),
            "obs_identifiers_unique": None,
            "var_identifiers_unique": None,
            "obs_columns": [],
            "condition_counts": [],
            "qc_fields": [],
            "candidate_replicate_fields": [],
            "replication_decision": replication,
        }
    else:
        with AUDIT.section("h5ad_storage_audit"):
            try:
                storage = inspect_h5ad_storage(INPUT_H5AD)
            except Exception as exc:
                storage_error = f"{type(exc).__name__}: {exc}"
                storage = {"X": placeholder_matrix("unavailable_without_loading_matrix", storage_error), "layers": [], "raw": {"present": False}, "top_level_keys": []}
            matrix_rows = [flatten_matrix_record("X", storage["X"])]
            if storage.get("raw", {}).get("present") and "X" in storage["raw"]:
                matrix_rows.append(flatten_matrix_record("raw/X", storage["raw"]["X"]))
            layer_rows = [flatten_matrix_record(row.get("layer", row.get("key", "layer")), row) for row in storage.get("layers", [])]
            write_csv("00_matrix_storage_audit", pd.DataFrame(matrix_rows))
            write_csv("00_layers_storage_audit", pd.DataFrame(layer_rows if layer_rows else [{"layer": None, "present": False}]))
            write_csv("00_raw_and_layers_availability", pd.DataFrame([{
                "raw_present": bool(storage.get("raw", {}).get("present")),
                "raw_keys": ";".join(storage.get("raw", {}).get("keys", [])),
                "n_layers": len(storage.get("layers", [])),
                "layer_names": ";".join(row.get("layer", "") for row in storage.get("layers", [])),
                "status": "available" if storage_error is None else "storage_audit_partial",
                "error": storage_error,
            }]))

        with AUDIT.section("obs_metadata_audit"):
            try:
                obs_metadata = inspect_obs_metadata(INPUT_H5AD, CONFIG)
            except Exception as exc:
                obs_error = f"{type(exc).__name__}: {exc}"
                obs_metadata = {
                    "shape": None,
                    "obs_names_unique": None,
                    "var_names_unique": None,
                    "n_duplicate_obs_names": None,
                    "n_duplicate_var_names": None,
                    "obs_columns": [],
                    "var_columns": [],
                    "qc_columns": [],
                    "replicate_candidate_columns": [],
                    "clade_candidate_columns": [],
                    "condition_candidate_columns": [],
                    "condition_counts": [],
                    "metadata_summary": [],
                }
            write_csv("obs_columns", pd.DataFrame([{"position": i, "column": col} for i, col in enumerate(obs_metadata["obs_columns"])]))
            write_csv("00_obs_columns", pd.DataFrame([{"position": i, "column": col} for i, col in enumerate(obs_metadata["obs_columns"])]))
            write_csv("00_var_columns", pd.DataFrame([{"position": i, "column": col} for i, col in enumerate(obs_metadata["var_columns"])]))
            write_csv("00_metadata_columns_summary", pd.DataFrame(obs_metadata["metadata_summary"]))
            write_csv("condition_counts", pd.DataFrame(obs_metadata["condition_counts"]))
            write_csv("00_condition_counts", pd.DataFrame(obs_metadata["condition_counts"]))
            write_csv("00_qc_fields", pd.DataFrame([{"column": col} for col in obs_metadata["qc_columns"]]))

        with AUDIT.section("replication_candidate_audit"):
            candidate_fields = candidate_replicate_audit(obs_metadata, CONFIG)
            write_csv("candidate_replicate_fields", candidate_fields)
            by_category = candidate_fields.groupby("field_category", observed=False)["field_name"].apply(lambda x: ";".join(x.astype(str))).reset_index(name="columns") if not candidate_fields.empty else pd.DataFrame(columns=["field_category", "columns"])
            write_csv("00_replicate_and_clade_metadata", by_category.rename(columns={"field_category": "metadata_type"}))
            replication = replication_decision(candidate_fields)
            write_csv("00_replication_decision", pd.DataFrame([replication]))

        with AUDIT.section("feasibility_assessment"):
            feasibility = feasibility_assessment(storage, CONFIG)
            write_csv("memory_feasibility", feasibility)
            write_csv("00_feasibility_assessment", feasibility)

        matrix_record = flatten_matrix_record("X", storage["X"])
        metadata_audit = {
            "status": "completed" if storage_error is None and obs_error is None else "completed_with_unavailable_properties",
            "input_h5ad": provenance,
            "shape": obs_metadata.get("shape") or storage["X"].get("shape"),
            "matrix": matrix_record,
            "raw": storage.get("raw"),
            "layers": [flatten_matrix_record(row.get("layer", row.get("key", "layer")), row) for row in storage.get("layers", [])],
            "obs_identifiers_unique": obs_metadata.get("obs_names_unique"),
            "var_identifiers_unique": obs_metadata.get("var_names_unique"),
            "n_duplicate_obs_names": obs_metadata.get("n_duplicate_obs_names"),
            "n_duplicate_var_names": obs_metadata.get("n_duplicate_var_names"),
            "obs_columns": obs_metadata.get("obs_columns", []),
            "var_columns": obs_metadata.get("var_columns", []),
            "condition_counts": obs_metadata.get("condition_counts", []),
            "qc_fields": obs_metadata.get("qc_columns", []),
            "candidate_replicate_fields": candidate_fields.to_dict(orient="records"),
            "replication_decision": replication,
            "storage_error": storage_error,
            "obs_metadata_error": obs_error,
        }

except MemoryLimitExceeded as exc:
    stopped_cleanly = True
    stop_reason = str(exc)
    write_csv("00_stop_reason", pd.DataFrame([{"status": "stopped_memory_threshold", "reason": stop_reason}]))
finally:
    current_rss = rss_bytes()
    memory_log = AUDIT.dataframe()
    write_csv("00_memory_rss_log", memory_log)
    peak_rss_bytes = max(AUDIT.peak_rss_bytes, current_rss)
    peak_rss_gb = bytes_to_gb(peak_rss_bytes)
    current_rss_gb = bytes_to_gb(current_rss)
    if not metadata_audit:
        metadata_audit = {
            "status": "stopped_before_completion",
            "input_h5ad": provenance,
            "replication_decision": replication,
            "stop_reason": stop_reason,
        }
    metadata_audit.update({
        "active_branch": branch,
        "required_branch": CONFIG["required_git_branch"],
        "current_process_rss_gb": current_rss_gb,
        "peak_process_rss_gb": peak_rss_gb,
        "memory_threshold_gb": bytes_to_gb(memory_threshold_bytes(CONFIG)),
        "phase1_transformations_run": [],
        "full_expression_matrix_densified": False,
        "source_h5ad_write_mode_used": False,
    })
    write_json(OUTPUT_DIR / "audit" / "metadata_audit.json", metadata_audit)
    provenance.update({
        "current_process_rss_gb": current_rss_gb,
        "peak_process_rss_gb": peak_rss_gb,
        "memory_threshold_gb": bytes_to_gb(memory_threshold_bytes(CONFIG)),
    })
    write_json(OUTPUT_DIR / "audit" / "provenance.json", provenance)
    summary = pd.DataFrame([{
        "input_h5ad": str(INPUT_H5AD),
        "input_exists": bool(provenance.get("exists", False)),
        "phase": CONFIG["phase"],
        "status": "stopped_cleanly" if stopped_cleanly else "completed",
        "stop_reason": stop_reason,
        "active_branch": branch,
        "current_rss_gb": current_rss_gb,
        "peak_rss_gb": peak_rss_gb,
        "memory_threshold_gb": bytes_to_gb(memory_threshold_bytes(CONFIG)),
        "dense_full_matrix_created": False,
        "normalization_run": False,
        "hvg_selection_run": False,
        "calculate_qc_metrics_run": False,
        "scgeo_run": False,
        "pca_run": False,
        "scanorama_run": False,
        "harmony_run": False,
        "scvi_run": False,
        "umap_run": False,
        "rna_velocity_run": False,
    }])
    write_csv("00_phase1_summary", summary)
    version_payload = {
        "packages": {name: __import__("gse249479_memory_safe").package_version(name) for name in CONFIG["version_record_packages"]},
        "current_process_rss_gb": current_rss_gb,
        "peak_process_rss_gb": peak_rss_gb,
    }
    write_json(OUTPUT_DIR / "version_records" / "00_metadata_replication_and_memory_audit_versions.json", version_payload)
    write_metadata(OUTPUT_DIR, CONFIG, PATHS, {
        "active_branch": branch,
        "input_h5ad": str(INPUT_H5AD),
        "input_exists": bool(provenance.get("exists", False)),
        "stopped_cleanly": bool(stopped_cleanly),
        "stop_reason": stop_reason,
        "current_process_rss_gb": current_rss_gb,
        "peak_process_rss_gb": peak_rss_gb,
        "replication_decision": replication,
    })
    gc.collect()

summary